# Automatic Sleep Scoring Comparison

Compare automatic sleep stagers (**YASA**, **GSSC**, **SleepFM**, and SleepFM recalibrations) against a manual hypnogram.

---
1. Select your **EDF folder** and **scoring folder**
2. Confirm the **channel mapping**
3. Pick which **stagers** to run
4. View **comparison plots** (hypnodensities, confusion matrices, agreement metrics)

_Launch this notebook with Voila: `bash autoscore.sh` or `voila score_edf_voila.ipynb`_

In [ ]:
#%% Imports and setup
import sys
import os
import json
import time
import warnings
import traceback
from pathlib import Path
from contextlib import redirect_stderr, redirect_stdout
import io as _io

os.environ['TQDM_DISABLE'] = '1'
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
mne.set_log_level('ERROR')
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from ipyfilechooser import FileChooser
    HAS_FILECHOOSER = True
except ImportError:
    HAS_FILECHOOSER = False

# Add local scoredf package
NOTEBOOK_DIR = Path.cwd() if '__file__' not in dir() else Path(__file__).parent
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from scoredf.io_score import detect_and_read_scoring, match_files, window_from_scoring
from scoredf.channels import (
    detect_channels, get_edf_channel_names, check_stager_requirements,
    rename_dict, ALIASES,
    load_remap_json, mapping_from_remap_entry,
)
from scoredf.plots import (
    plot_hypnodensities, plot_confusion_matrices, compute_agreement,
    plot_agreement_summary, plot_hypnogram_grid, STAGES, STAGE_COLORS, STAGER_COLORS,
)

from sleephd.io import NapWindow, STAGE_NAMES
from sleephd.alignment import (
    build_epoch_table, aggregate_hypnodensity, make_nap_epochs, align_manual_hypnogram,
)
from sleephd.signals import EEG_DERIVATIONS

# Electrode -> derivation lookup (e.g. C3 -> C3-M2)
ELECTRODE_TO_DERIVATION = {v[0]: k for k, v in EEG_DERIVATIONS.items()}
CONTRALATERAL = {'C3': 'C4', 'C4': 'C3', 'F3': 'F4', 'F4': 'F3', 'O1': 'O2', 'O2': 'O1'}

# Probe stager availability
GSSC_AVAILABLE = False
try:
    from gssc.infer import EEGInfer  # noqa: F401
    GSSC_AVAILABLE = True
except ImportError:
    pass

YASA_AVAILABLE = False
try:
    import yasa  # noqa: F401
    YASA_AVAILABLE = True
except ImportError:
    pass

SLEEPFM_AVAILABLE = False
try:
    from sleephd.sleepfm import _default_sleepfm_root
    SLEEPFM_AVAILABLE = (_default_sleepfm_root() / 'sleepfm' / 'checkpoints').exists()
except Exception:
    pass

ALL_STAGERS = ['YASA', 'GSSC', 'SleepFM', 'SleepFM recal A', 'SleepFM recal B']

# Global state
STATE = {}

print('Packages loaded.')
avail = []
if YASA_AVAILABLE: avail.append('YASA')
if GSSC_AVAILABLE: avail.append('GSSC')
if SLEEPFM_AVAILABLE: avail.append('SleepFM')
print(f'Available stagers: {avail if avail else "none (install packages in sleephd env)"}')

#%% ============================================================
#    PHASE 1: Folder selection and file scanning
#    ============================================================
display(HTML('<hr style="height:4px; background-color:black; border:none;"><h2>1. Select your data folders</h2>'))

if HAS_FILECHOOSER:
    chooser_edf = FileChooser(os.getcwd())
    chooser_edf.title = '<b>EDF folder</b>'
    chooser_edf.show_only_dirs = True

    chooser_scoring = FileChooser(os.getcwd())
    chooser_scoring.title = '<b>Scoring folder</b>'
    chooser_scoring.show_only_dirs = True
else:
    chooser_edf = widgets.Text(description='EDF folder:', layout=widgets.Layout(width='80%'))
    chooser_scoring = widgets.Text(description='Scoring folder:', layout=widgets.Layout(width='80%'))

scan_button = widgets.Button(description='Scan folders', button_style='success', icon='search')
scan_output = widgets.Output()

# Phase 2 widgets (created after scan)
phase2_box = widgets.VBox()
run_button = widgets.Button(description='Run scoring', button_style='primary', icon='play')
run_output = widgets.Output()
results_output = widgets.Output()


def get_folder(chooser):
    if HAS_FILECHOOSER:
        return chooser.selected_path or chooser.value
    return chooser.value.strip()


def _find_remap_json(edf_folder):
    """Search for a remap JSON from notebook 2 in config_param/ next to the EDF folder."""
    candidates = [
        Path(edf_folder).parent / 'config_param' / 'remap_reref_persubject.json',
        Path(edf_folder) / 'config_param' / 'remap_reref_persubject.json',
        Path(edf_folder).parent / 'remap_reref_persubject.json',
    ]
    for p in candidates:
        if p.is_file():
            return p
    return None


def on_scan_click(_):
    scan_output.clear_output()
    with scan_output:
        edf_folder = get_folder(chooser_edf)
        scoring_folder = get_folder(chooser_scoring)
        if not edf_folder or not Path(edf_folder).is_dir():
            print('Please select a valid EDF folder.')
            return
        if not scoring_folder or not Path(scoring_folder).is_dir():
            print('Please select a valid scoring folder.')
            return

        print(f'EDF folder:     {edf_folder}')
        print(f'Scoring folder: {scoring_folder}')
        print()

        # Match files (prefers remapped hypnograms from notebook 3 when available)
        matched, unmatched_edfs, unmatched_scorings = match_files(edf_folder, scoring_folder)
        STATE['matched'] = matched
        STATE['edf_folder'] = edf_folder
        STATE['scoring_folder'] = scoring_folder

        if not matched:
            print('No EDF-scoring matches found. Check that filenames share a common stem.')
            return

        n_remapped = sum(1 for m in matched if m.get('scoring_remapped'))
        print(f'Matched recordings: {len(matched)}')
        if n_remapped:
            print(f'  ({n_remapped} using remapped hypnograms from notebook 3)')
        for m in matched:
            tag = ' [remapped]' if m.get('scoring_remapped') else ''
            print(f'  {m["stem"]:20s}  EDF: {m["edf"].name:30s}  Scoring: {m["scoring"].name}{tag}')
        if unmatched_edfs:
            print(f'\nUnmatched EDFs ({len(unmatched_edfs)}):')
            for f in unmatched_edfs:
                print(f'  {f.name}')
        if unmatched_scorings:
            print(f'\nUnmatched scoring files ({len(unmatched_scorings)}):')
            for f in unmatched_scorings:
                print(f'  {f.name}')

        # Look for remap JSON from notebook 2
        remap_json_path = _find_remap_json(edf_folder)
        remap_json_data = None
        matched_stems = {m['stem'] for m in matched}

        if remap_json_path is not None:
            try:
                remap_json_data = load_remap_json(remap_json_path)
                covered = matched_stems & set(remap_json_data.keys())
                missing_subs = matched_stems - set(remap_json_data.keys())
                print(f'\nFound channel remap JSON (notebook 2): {remap_json_path}')
                print(f'  Covers {len(covered)}/{len(matched_stems)} matched subjects.')
                if missing_subs:
                    print(f'  Not in JSON: {sorted(missing_subs)}')
            except Exception as e:
                print(f'\nFound channel remap JSON but failed to load: {e}')
                remap_json_data = None

        STATE['remap_json_path'] = remap_json_path
        STATE['remap_json_data'] = remap_json_data

        # Scan all EDF headers, group subjects by channel configuration
        print('\nScanning EDF channels...')
        configs_raw = {}
        stem_to_config_key = {}
        stem_to_ch_names = {}
        for m in matched:
            ch_names = get_edf_channel_names(m['edf'])
            stem_to_ch_names[m['stem']] = ch_names
            config_key = tuple(sorted(ch_names))
            if config_key not in configs_raw:
                auto = detect_channels(m['edf'])
                configs_raw[config_key] = {
                    'stems': [],
                    'ch_names': ch_names,
                    'auto_mapping': auto,
                }
            configs_raw[config_key]['stems'].append(m['stem'])
            stem_to_config_key[m['stem']] = config_key

        labeled_configs = {}
        config_key_to_label = {}
        for i, (key, cfg) in enumerate(configs_raw.items(), 1):
            label = f'config. {i} (n={len(cfg["stems"])})'
            labeled_configs[label] = cfg
            config_key_to_label[key] = label

        stem_to_config = {
            stem: config_key_to_label[key]
            for stem, key in stem_to_config_key.items()
        }

        STATE['configs'] = labeled_configs
        STATE['stem_to_config'] = stem_to_config
        STATE['stem_to_ch_names'] = stem_to_ch_names

        print(f'Found {len(labeled_configs)} channel configuration(s):')
        for label, cfg in labeled_configs.items():
            detected = {k: v for k, v in cfg['auto_mapping'].items() if v is not None}
            print(f'  {label}: {len(cfg["ch_names"])} channels, '
                  f'auto-detected {len(detected)}/{len(ALIASES)} standard roles')
            print(f'    Subjects: {sorted(cfg["stems"])}')

    # Build Phase 2 UI
    _build_phase2(matched, remap_json_path, remap_json_data)


def _build_phase2(matched, remap_json_path, remap_json_data):
    matched_stems = {m['stem'] for m in matched}
    labeled_configs = STATE['configs']

    # --- Channel source toggle (JSON vs manual) ---
    json_usable = (
        remap_json_data is not None
        and matched_stems <= set(remap_json_data.keys())
    )
    json_available_partial = (
        remap_json_data is not None
        and not json_usable
        and (matched_stems & set(remap_json_data.keys()))
    )

    use_json_toggle = widgets.RadioButtons(
        options=[
            ('Use remap JSON from notebook 2 (recommended)', 'json'),
            ('Manual mapping below', 'manual'),
        ],
        value='json' if json_usable else 'manual',
        layout=widgets.Layout(width='auto'),
    )
    STATE['use_json_toggle'] = use_json_toggle

    if json_usable:
        json_info = widgets.HTML(
            f'<p style="color:green;">Channel remap JSON covers all {len(matched_stems)} subjects: '
            f'<code>{remap_json_path}</code></p>'
        )
    elif json_available_partial:
        missing_subs = sorted(matched_stems - set(remap_json_data.keys()))
        json_info = widgets.HTML(
            f'<p style="color:orange;">Channel remap JSON found but missing subjects: {missing_subs}. '
            f'Those will need manual mapping below. <code>{remap_json_path}</code></p>'
        )
    elif remap_json_data is not None:
        json_info = widgets.HTML(
            '<p style="color:red;">Channel remap JSON found but covers none of the matched subjects.</p>'
        )
    else:
        json_info = widgets.HTML(
            '<p style="color:gray;">No channel remap JSON found (from notebook 2: '
            '<i>Select &amp; remap channels</i>). Please map channels manually below.</p>'
        )

    # --- Per-configuration manual mapping (Accordion) ---
    config_dropdowns = {}
    accordion_children = []
    accordion_titles = []

    for cfg_label, cfg in labeled_configs.items():
        ch_names = cfg['ch_names']
        auto_mapping = cfg['auto_mapping']
        options = ['-- not available --'] + sorted(ch_names)

        dropdowns = {}
        rows = []
        for standard in ALIASES:
            current = auto_mapping.get(standard)
            dd = widgets.Dropdown(
                options=options,
                value=current if current in ch_names else '-- not available --',
                layout=widgets.Layout(width='250px'),
            )
            dropdowns[standard] = dd
            rows.append(
                widgets.HBox([
                    widgets.Label(f'{standard}:', layout=widgets.Layout(width='100px')),
                    dd,
                ])
            )
        config_dropdowns[cfg_label] = dropdowns

        subjects_html = widgets.HTML(
            f'<p style="color:gray; font-size:0.9em;">'
            f'Subjects: {", ".join(sorted(cfg["stems"]))}<br>'
            f'EDF channels: {", ".join(sorted(ch_names))}</p>'
        )
        panel = widgets.VBox([subjects_html] + rows)
        accordion_children.append(panel)
        accordion_titles.append(cfg_label)

    STATE['config_dropdowns'] = config_dropdowns

    manual_accordion = widgets.Accordion(children=accordion_children)
    for i, title in enumerate(accordion_titles):
        manual_accordion.set_title(i, title)
    manual_accordion.selected_index = 0

    manual_channel_box = widgets.VBox([
        widgets.HTML(
            '<p>For each channel configuration, assign the EDF channel that '
            'corresponds to each standard role. The initial values are '
            'auto-detected guesses &mdash; <b>please verify</b>.</p>'
        ),
        manual_accordion,
    ])

    def _on_toggle_change(change):
        manual_channel_box.layout.display = 'none' if change['new'] == 'json' else ''

    use_json_toggle.observe(_on_toggle_change, names='value')
    if not json_usable and remap_json_data is None:
        use_json_toggle.layout.display = 'none'
    manual_channel_box.layout.display = 'none' if use_json_toggle.value == 'json' else ''

    channel_box = widgets.VBox([
        widgets.HTML('<h3>Channel mapping</h3>'),
        json_info,
        use_json_toggle,
        manual_channel_box,
    ])

    # Stager checkboxes
    checkboxes = {}
    stager_rows = []
    n_recordings = len(matched)

    for name in ALL_STAGERS:
        enabled = True
        note = ''

        if name == 'YASA' and not YASA_AVAILABLE:
            enabled = False
            note = ' (yasa package not installed)'
        elif name == 'GSSC' and not GSSC_AVAILABLE:
            enabled = False
            note = ' (gssc package not installed)'
        elif name.startswith('SleepFM') and not SLEEPFM_AVAILABLE:
            enabled = False
            note = ' (SleepFM checkpoints not found)'

        if name == 'SleepFM recal A' and n_recordings < 2:
            enabled = False
            note = f' (needs >= 2 recordings, have {n_recordings})'
        if name == 'SleepFM recal B' and n_recordings < 3:
            enabled = False
            note = f' (needs >= 3 recordings, have {n_recordings})'

        cb = widgets.Checkbox(
            value=enabled,
            description=name + note,
            disabled=not enabled,
            indent=False,
            layout=widgets.Layout(width='400px'),
        )
        checkboxes[name] = cb
        stager_rows.append(cb)

    STATE['checkboxes'] = checkboxes

    if n_recordings < 6 and SLEEPFM_AVAILABLE:
        stager_rows.append(widgets.HTML(
            f'<p style="color:orange;">Note: recal B works best with >= 6 recordings (you have {n_recordings}).</p>'
        ))

    # Preferred EEG electrode dropdowns
    central_dd = widgets.Dropdown(
        options=['C3', 'C4'], value='C3',
        layout=widgets.Layout(width='100px'),
    )
    frontal_dd = widgets.Dropdown(
        options=['F3', 'F4'], value='F4',
        layout=widgets.Layout(width='100px'),
    )
    occipital_dd = widgets.Dropdown(
        options=['O1', 'O2'], value='O2',
        layout=widgets.Layout(width='100px'),
    )
    STATE['electrode_picks'] = {
        'central': central_dd,
        'frontal': frontal_dd,
        'occipital': occipital_dd,
    }

    electrode_box = widgets.VBox([
        widgets.HTML('<h4>Preferred EEG electrodes</h4>'),
        widgets.HTML(
            '<p style="font-size:0.9em;">Pick one electrode per region. '
            'The derivation (e.g. C3&#8722;M2) is computed automatically.<br>'
            'If the preferred electrode is unavailable for a subject, '
            'the contralateral side is tried.</p>'
        ),
        widgets.HBox([
            widgets.Label('Central:', layout=widgets.Layout(width='80px')), central_dd,
            widgets.HTML('&nbsp;&nbsp;'),
            widgets.Label('Frontal:', layout=widgets.Layout(width='80px')), frontal_dd,
            widgets.HTML('&nbsp;&nbsp;'),
            widgets.Label('Occipital:', layout=widgets.Layout(width='80px')), occipital_dd,
        ]),
    ])

    stager_box = widgets.VBox(
        [widgets.HTML('<h3>Stagers to run</h3>')] + stager_rows + [electrode_box],
    )

    # Force recompute checkbox
    force_cb = widgets.Checkbox(value=False, description='Force recompute (ignore cache)', indent=False)
    STATE['force_cb'] = force_cb

    phase2_box.children = [
        widgets.HTML('<hr style="height:4px; background-color:black; border:none;"><h2>2. Confirm channels and select stagers</h2>'),
        widgets.HBox([channel_box, widgets.HTML('&nbsp;' * 10), stager_box]),
        force_cb,
        run_button,
        run_output,
        results_output,
    ]


scan_button.on_click(on_scan_click)
display(chooser_edf, chooser_scoring, scan_button, scan_output, phase2_box)


#%% ============================================================
#    PHASE 3: Run stagers
#    ============================================================
def _get_config_mapping(cfg_label):
    """Read the manual dropdown mapping for a given configuration label."""
    dropdowns = STATE['config_dropdowns'][cfg_label]
    mapping = {}
    for standard, dd in dropdowns.items():
        val = dd.value
        mapping[standard] = val if val != '-- not available --' else None
    return mapping


def _get_rec_mapping(sid, edf_path):
    """Return the channel mapping for a single recording.

    Uses the remap JSON when the user selected that mode and the subject is in
    the JSON, otherwise reads the manual per-configuration dropdowns.
    """
    use_json = STATE.get('use_json_toggle') and STATE['use_json_toggle'].value == 'json'
    remap_json_data = STATE.get('remap_json_data')

    if use_json and remap_json_data and sid in remap_json_data:
        entry = remap_json_data[sid]
        return mapping_from_remap_entry(entry['remap'])

    cfg_label = STATE['stem_to_config'][sid]
    return _get_config_mapping(cfg_label)


def _available_after_rename(edf_ch_names, rec_rename):
    """Standard channel names that will actually exist after renaming."""
    return {rec_rename.get(ch, ch) for ch in edf_ch_names}


def _derivation_ok(available, derivation_name):
    """True when both electrodes of a derivation are in the available set."""
    active, ref = EEG_DERIVATIONS[derivation_name]
    return active in available and ref in available


def _resolve_yasa_eeg(available):
    """Pick the YASA EEG derivation: preferred central, then contralateral."""
    pref = STATE['electrode_picks']['central'].value
    deriv = ELECTRODE_TO_DERIVATION[pref]
    if _derivation_ok(available, deriv):
        return deriv
    alt_deriv = ELECTRODE_TO_DERIVATION[CONTRALATERAL[pref]]
    if _derivation_ok(available, alt_deriv):
        return alt_deriv
    return None


def _resolve_gssc_eeg(available):
    """Build the GSSC EEG list from available central + frontal derivations."""
    candidates = []
    for role in ('central', 'frontal'):
        pref = STATE['electrode_picks'][role].value
        candidates.append(ELECTRODE_TO_DERIVATION[pref])
        candidates.append(ELECTRODE_TO_DERIVATION[CONTRALATERAL[pref]])
    return tuple(d for d in dict.fromkeys(candidates) if _derivation_ok(available, d))


def on_run_click(_):
    run_output.clear_output()
    results_output.clear_output()

    with run_output:
        matched = STATE.get('matched', [])
        if not matched:
            print('No matched recordings. Scan folders first.')
            return

        selected_stagers = [name for name, cb in STATE['checkboxes'].items() if cb.value]
        force = STATE['force_cb'].value

        if not selected_stagers:
            print('No stagers selected.')
            return

        use_json = STATE.get('use_json_toggle') and STATE['use_json_toggle'].value == 'json'
        if use_json:
            print(f'Channel source: remap JSON ({STATE.get("remap_json_path")})')
        else:
            print('Channel source: manual mapping')
            for cfg_label in STATE['configs']:
                m = _get_config_mapping(cfg_label)
                assigned = {k: v for k, v in m.items() if v is not None}
                print(f'  {cfg_label}: {assigned}')

        # Show electrode preferences
        picks = {role: dd.value for role, dd in STATE['electrode_picks'].items()}
        derivs = {role: ELECTRODE_TO_DERIVATION[el] for role, el in picks.items()}
        print(f'Preferred electrodes: C={picks["central"]} ({derivs["central"]}), '
              f'F={picks["frontal"]} ({derivs["frontal"]}), '
              f'O={picks["occipital"]} ({derivs["occipital"]})')

        # Pre-flight: check at least one config can run each stager
        for name in list(selected_stagers):
            can_run = False
            for cfg_label, cfg in STATE['configs'].items():
                if use_json:
                    first_stem = cfg['stems'][0]
                    first_edf = next(m['edf'] for m in matched if m['stem'] == first_stem)
                    check_mapping = _get_rec_mapping(first_stem, first_edf)
                else:
                    check_mapping = _get_config_mapping(cfg_label)
                rn = rename_dict(check_mapping)
                avail = _available_after_rename(cfg['ch_names'], rn)

                if name == 'YASA':
                    ok = _resolve_yasa_eeg(avail) is not None
                    if not ok:
                        print(f'YASA: {cfg_label} has no usable central EEG derivation')
                elif name == 'GSSC':
                    ok = bool(_resolve_gssc_eeg(avail)) and 'hEOG' in avail
                    if not ok:
                        print(f'GSSC: {cfg_label} missing EEG derivations or EOG')
                else:
                    ok, missing = check_stager_requirements(check_mapping, name)
                    if not ok:
                        print(f'{name}: {cfg_label} missing channels {missing}')

                if ok:
                    can_run = True
            if not can_run:
                print(f'{name}: no configuration has the required channels -- skipping.')
                selected_stagers.remove(name)
        if not selected_stagers:
            print('No stagers can run with the current channel mapping.')
            return

        # Determine output directory
        out_dir = Path(STATE['edf_folder']).parent / 'autoscore_outputs'
        cache_dir = out_dir / 'cache'
        cache_dir.mkdir(parents=True, exist_ok=True)
        print(f'Output directory: {out_dir}')
        print(f'Selected stagers: {selected_stagers}')
        print(f'Recordings: {len(matched)}')
        print()

        # Progress
        progress = widgets.IntProgress(
            min=0, max=len(matched), description='Scoring:', bar_style='info',
            layout=widgets.Layout(width='80%'),
        )
        display(progress)

        # Per-recording data
        five_second = {name: {} for name in selected_stagers}
        recordings_info = {}
        all_frames = []
        failed = []

        for rec_idx, rec in enumerate(matched):
            sid = rec['stem']
            edf_path = rec['edf']
            scoring_path = rec['scoring']
            progress.value = rec_idx
            progress.description = f'{sid}:'
            print(f'--- {sid} ({rec_idx+1}/{len(matched)}) ---')

            try:
                # Read scoring
                hypnogram, epoch_seconds = detect_and_read_scoring(scoring_path)
                src_tag = ' [remapped]' if rec.get('scoring_remapped') else ''
                print(f'  Scoring: {scoring_path.name}{src_tag} ({len(hypnogram)} epochs, {epoch_seconds}s)')

                # Derive window
                window = window_from_scoring(hypnogram, edf_path, epoch_seconds)
                print(f'  Window: {window.lights_off_clock} -- {window.reveil_clock} '
                      f'({window.duration_seconds/60:.1f} min)')

                recordings_info[sid] = {
                    'edf': edf_path, 'window': window,
                    'hypnogram': hypnogram, 'epoch_seconds': epoch_seconds,
                }

                rec_mapping = _get_rec_mapping(sid, edf_path)
                rec_rename = rename_dict(rec_mapping)
                edf_ch_names = STATE['stem_to_ch_names'][sid]
                available = _available_after_rename(edf_ch_names, rec_rename)

                # --- YASA ---
                if 'YASA' in selected_stagers:
                    yasa_eeg = _resolve_yasa_eeg(available)
                    if yasa_eeg:
                        yasa_eog = 'hEOG' if 'hEOG' in available else None
                        yasa_emg = 'EMG' if 'EMG' in available else None
                        print(f'  Running YASA (EEG={yasa_eeg}'
                              f'{", EOG=" + yasa_eog if yasa_eog else ""}'
                              f'{", EMG=" + yasa_emg if yasa_emg else ""})...')
                        from sleephd.stagers import infer_yasa
                        five_second['YASA'][sid] = infer_yasa(
                            edf_path, window,
                            eeg=yasa_eeg,
                            eog=yasa_eog,
                            emg=yasa_emg,
                            cache_path=cache_dir / f'{sid}_yasa.csv',
                            force=force,
                            rename=rec_rename or None,
                        )
                    else:
                        print(f'  YASA: skipped (no central EEG derivation for {sid})')

                # --- GSSC ---
                if 'GSSC' in selected_stagers:
                    gssc_eeg = _resolve_gssc_eeg(available)
                    if gssc_eeg and 'hEOG' in available:
                        print(f'  Running GSSC ({", ".join(gssc_eeg)})...')
                        from sleephd.stagers import infer_gssc
                        with redirect_stderr(_io.StringIO()):
                            five_second['GSSC'][sid] = infer_gssc(
                                edf_path, window,
                                eeg=gssc_eeg,
                                cache_path=cache_dir / f'{sid}_gssc.csv',
                                force=force,
                                rename=rec_rename or None,
                            )
                    else:
                        reason = 'no EEG derivations' if not gssc_eeg else 'missing EOG'
                        print(f'  GSSC: skipped ({reason} for {sid})')

                # --- SleepFM ---
                needs_sleepfm = any(
                    s in selected_stagers
                    for s in ['SleepFM', 'SleepFM recal A', 'SleepFM recal B']
                )
                if needs_sleepfm:
                    ok, _ = check_stager_requirements(rec_mapping, 'SleepFM')
                    if ok:
                        print('  Running SleepFM...')
                        from sleephd.signals import infer_sleepfm_montage
                        sfm_result = infer_sleepfm_montage(
                            edf_path, cache_dir,
                            eeg_derivations=list(EEG_DERIVATIONS),
                            cache_tag='aasm_all_eeg',
                            include_eog=True,
                            device='cpu',
                            cover_tail=True,
                            force=force,
                            rename=rec_rename or None,
                        )
                        if 'SleepFM' in selected_stagers:
                            five_second['SleepFM'][sid] = sfm_result
                        recordings_info[sid]['sleepfm_5s'] = sfm_result
                    else:
                        print(f'  SleepFM: skipped (missing channels for {sid})')

            except Exception as e:
                print(f'  ERROR: {e}')
                traceback.print_exc()
                failed.append((sid, str(e)))

        progress.value = len(matched)
        progress.description = 'Done:'
        progress.bar_style = 'success'

        # --- Recalibration (LOSO) ---
        recal_subjects = [
            sid for sid in recordings_info
            if 'sleepfm_5s' in recordings_info[sid]
        ]
        needs_recal = any(
            s in selected_stagers for s in ['SleepFM recal A', 'SleepFM recal B']
        )
        base_recs = None
        if needs_recal and len(recal_subjects) >= 2:
            from sleephd import recalibration
            base_recs = []
            for sid in recal_subjects:
                info = recordings_info[sid]
                base_recs.append(recalibration.prepare_recording(
                    sid, info['sleepfm_5s'], info['window'], info['hypnogram'], info['epoch_seconds'],
                ))

        if 'SleepFM recal A' in selected_stagers and base_recs is not None and len(base_recs) >= 2:
            print('\n--- Recalibration A (LOSO) ---')
            five_second['SleepFM recal A'] = {}
            for held_out in base_recs:
                train = [r for r in base_recs if r.subject != held_out.subject]
                params = recalibration.fit(train)
                five_second['SleepFM recal A'][held_out.subject] = (
                    recalibration.recalibrated_five_second(held_out, params)
                )
                print(f'  {held_out.subject}: T={params["temperature"]:.3f}, converged={params["converged"]}')

        if 'SleepFM recal B' in selected_stagers and base_recs is not None and len(base_recs) >= 3:
            print('\n--- Recalibration B (LOSO, refit fc) ---')
            from sleephd.sleepfm import head_features as _head_features
            from scipy.special import softmax

            heads = []
            fc_weight = fc_bias = None
            for rec_base in base_recs:
                sid = rec_base.subject

                feat_path = cache_dir / f'{sid}_head_features.npz'
                if not feat_path.exists() or force:
                    primary = _head_features(
                        cache_dir / f'{sid}_aasm_all_eeg_128hz.hdf5'
                    )
                    features = primary['features']
                    extra = len(rec_base.five_second) - len(features)
                    if extra > 0:
                        tail_path = cache_dir / f'{sid}_aasm_all_eeg_tail_128hz.hdf5'
                        if tail_path.exists():
                            tail = _head_features(tail_path)
                            features = np.vstack([features, tail['features'][-extra:]])
                    np.savez(feat_path, features=features,
                             fc_weight=primary['fc_weight'], fc_bias=primary['fc_bias'])

                data = np.load(feat_path)
                fc_weight = data['fc_weight']
                fc_bias = data['fc_bias']
                heads.append(recalibration.prepare_head_recording(rec_base, data['features']))
                print(f'  {sid}: {data["features"].shape[0]} feature steps')

            five_second['SleepFM recal B'] = {}
            inner_folds = min(5, len(heads) - 1) if len(heads) > 2 else len(heads) - 1
            for held_out in heads:
                train = [h for h in heads if h.subject != held_out.subject]
                result = recalibration.fit_fc_nested(
                    train, fc_weight, fc_bias, inner_folds=max(2, inner_folds),
                )
                five_second['SleepFM recal B'][held_out.subject] = (
                    recalibration.refit_five_second(held_out, result['weight'], result['bias'])
                )
                print(f'  {held_out.subject}: lambda={result["penalty"]}')

        # --- Build unified epoch table ---
        print('\nBuilding epoch tables...')
        active_stagers = [s for s in selected_stagers if five_second.get(s)]
        frames = []
        for name in active_stagers:
            for sid in five_second[name]:
                info = recordings_info[sid]
                table = build_epoch_table(
                    info['window'], None, info['hypnogram'],
                    hypnodensity_5s=five_second[name][sid],
                    epoch_seconds=info['epoch_seconds'],
                )
                table['minutes_from_start'] = (
                    (table['start_seconds'] - info['window'].lights_off_seconds) / 60
                )
                frames.append(table.assign(subject=sid, stager=name))

        if not frames:
            print('No results to display.')
            return

        epochs_df = pd.concat(frames, ignore_index=True)
        scored_df = epochs_df.loc[epochs_df['manual_stage'] != 'Unknown']
        subjects = sorted(recordings_info.keys())

        # Save
        epochs_df.to_csv(out_dir / 'epochs_all_stagers.csv', index=False)
        print(f'Saved epoch table to {out_dir / "epochs_all_stagers.csv"}')

        STATE['epochs_df'] = epochs_df
        STATE['scored_df'] = scored_df
        STATE['active_stagers'] = active_stagers
        STATE['subjects'] = subjects
        STATE['out_dir'] = out_dir

        if failed:
            print(f'\nFailed recordings ({len(failed)}):')
            for sid, err in failed:
                print(f'  {sid}: {err}')

        print('\nDone! Generating results...')

    # Generate results
    _show_results()


#%% ============================================================
#    PHASE 4: Results display
#    ============================================================
def _show_results():
    results_output.clear_output()
    with results_output:
        epochs_df = STATE['epochs_df']
        scored_df = STATE['scored_df']
        stagers = STATE['active_stagers']
        subjects = STATE['subjects']
        out_dir = STATE['out_dir']

        display(HTML('<hr style="height:4px; background-color:black; border:none;"><h2>3. Results</h2>'))

        # --- Agreement table ---
        display(HTML('<h3>3.1 Agreement with manual hypnogram</h3>'))
        agreement = compute_agreement(scored_df, stagers, subjects)
        agreement.to_csv(out_dir / 'agreement_summary.csv')
        display(agreement.round(3).style.background_gradient(cmap='Greens', subset=['accuracy', 'kappa_pooled']))

        # --- Agreement figure ---
        try:
            fig = plot_agreement_summary(agreement, stagers, subjects)
            fig.savefig(out_dir / 'agreement_by_stager.png', dpi=130, bbox_inches='tight')
            display(fig)
            plt.close(fig)
        except Exception as e:
            print(f'Agreement plot error: {e}')

        # --- Confusion matrices ---
        display(HTML('<h3>3.2 Confusion matrices</h3>'))
        try:
            fig = plot_confusion_matrices(scored_df, stagers)
            fig.savefig(out_dir / 'confusion_by_stager.png', dpi=130, bbox_inches='tight')
            display(fig)
            plt.close(fig)
        except Exception as e:
            print(f'Confusion plot error: {e}')

        # --- Hypnogram grid ---
        display(HTML('<h3>3.3 Hypnogram comparison</h3>'))
        try:
            fig = plot_hypnogram_grid(epochs_df, stagers, subjects)
            fig.savefig(out_dir / 'hypnograms_by_stager.png', dpi=130, bbox_inches='tight')
            display(fig)
            plt.close(fig)
        except Exception as e:
            print(f'Hypnogram grid error: {e}')

        # --- Per-recording hypnodensities ---
        display(HTML('<h3>3.4 Hypnodensities per recording</h3>'))
        for sid in subjects:
            try:
                fig = plot_hypnodensities(epochs_df, stagers, sid)
                fig.savefig(out_dir / f'hypnodensities_{sid}.png', dpi=130, bbox_inches='tight')
                display(fig)
                plt.close(fig)
            except Exception as e:
                print(f'Hypnodensity plot error for {sid}: {e}')

        display(HTML(f'<hr><p>All outputs saved to <code>{out_dir}</code></p>'))


run_button.on_click(on_run_click)